In [ ]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import sys
import re
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'

def levenshtein_distance(s1, s2):
    """

    Возвращает:
        int: расстояние Левенштейна.
    """
    # Создаём матрицу (len(s1)+1) x (len(s2)+1)
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # Инициализация первой строки и первого столбца
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    # Заполнение матрицы
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],     # удаление
                    dp[i][j - 1],     # вставка
                    dp[i - 1][j - 1]  # замена
                )

    return dp[m][n]

def remove_white_noise(image, method, kernelSize, sigmaa):
    """
    Удаляет белый шум (соль) с изображения.
    
    Параметры:
        image : numpy.ndarray (обычно grayscale)
        method : str
            'median'      — медианный фильтр (хорош для шума типа «соль и перец»)
            'morph_open'  — морфологическое открытие (удаляет мелкие белые объекты)
            'gaussian'    — гауссово размытие (сглаживает, но размывает края)
        kwargs : передаются конкретному методу:
            kernel_size (int) : размер ядра (по умолчанию 3 для median/gaussian, 2 для morph_open)
            sigma (float) : для гауссова размытия (по умолчанию 0)
    
    Возвращает:
        numpy.ndarray — изображение после удаления шума.
    """
    if method == 'median':
        ksize = kernelSize
        # cv2.medianBlur работает только с нечётным размером ядра
        if ksize % 2 == 0:
            ksize += 1
        return cv2.medianBlur(image, ksize)
    elif method == 'morph_open':
        kernel_size = kernelSize
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        return cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
    elif method == 'gaussian':
        ksize = kernelSize
        if ksize % 2 == 0:
            ksize += 1
        sigma = sigmaa
        return cv2.GaussianBlur(image, (ksize, ksize), sigma)
    else:
        raise ValueError(f"Неизвестный метод: {method}")

def display_result(img, text, title):

    plt.imshow(img)

    plt.axis('off') 

    plt.show()
    #print(f'\n{"="*10}display_result{"="*10}\n')

def preprocess_image(image, mint, maxt, ker, ker2, sig, stri):
    """
    Загружает изображение, переводит в оттенки серого,
    применяет фиксированный порог и удаляет белый шум морфологией.
    Возвращает очищенное бинарное изображение.
    """
    img = image #cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Не удалось загрузить изображение: {image_path}")
    print(f'\n{"="*60}\nimg: \n{"="*60}\n')
    display_result(img, "1", "img")
    gray = img #cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #display_result(gray, "2", "gray")
    _, binary = cv2.threshold(gray, mint, maxt, cv2.THRESH_BINARY)
    print(f'\n{"="*60}\nbinary: \n{"="*60}\n')
    display_result(binary, "3", "binary")
    #kernel = np.ones((ker, ker), np.uint8)
    denoised = remove_white_noise(binary, stri, ker2, sig)#cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    print(f'\n{"="*60}\ndenoised: \n{"="*60}\n')
    display_result(denoised, "4", "denoised")
    #denoised1 = remove_white_noise(denoised, stri, ker2, sig)
    #display_result(denoised1, "5", "denoised1")
    return denoised

def no_preprocess_image(image, mint, maxt, ker, ker2, sig, stri):
    """
    Загружает изображение, переводит в оттенки серого,
    применяет фиксированный порог и удаляет белый шум морфологией.
    Возвращает очищенное бинарное изображение.
    """
    img = image #cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Не удалось загрузить изображение: {image_path}")
    #print(f'\n{"="*60}\nimg: \n{"="*60}\n')
    #display_result(img, "1", "img")
    gray = img #cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #display_result(gray, "2", "gray")
    #_, binary = cv2.threshold(gray, mint, maxt, cv2.THRESH_BINARY)
    #print(f'\n{"="*60}\nbinary: \n{"="*60}\n')
    #display_result(binary, "3", "binary")
    #kernel = np.ones((ker, ker), np.uint8)
    denoised = gray #remove_white_noise(binary, stri, ker2, sig)#cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    print(f'\n{"="*60}\ndenoised: \n{"="*60}\n')
    display_result(denoised, "4", "denoised")
    #denoised1 = remove_white_noise(denoised, stri, ker2, sig)
    #display_result(denoised1, "5", "denoised1")
    return denoised

def denoise_binary_preprocess_image(image, mint, maxt, ker, ker2, sig, stri):
    """
    Загружает изображение, переводит в оттенки серого,
    применяет фиксированный порог и удаляет белый шум морфологией.
    Возвращает очищенное бинарное изображение.
    """
    img = image #cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Не удалось загрузить изображение: {image_path}")
    print(f'\n{"="*60}\nimg: \n{"="*60}\n')
    display_result(img, "1", "img")
    gray = img #cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #display_result(gray, "2", "gray")
    #kernel = np.ones((ker, ker), np.uint8)
    denoised = remove_white_noise(gray, stri, ker2, sig)#cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    print(f'\n{"="*60}\ndenoised: \n{"="*60}\n')
    display_result(denoised, "3", "denoised")
    # ===========================================
    _, binary = cv2.threshold(denoised, mint, maxt, cv2.THRESH_BINARY)
    print(f'\n{"="*60}\nbinary: \n{"="*60}\n')
    display_result(binary, "4", "binary")
    #denoised1 = remove_white_noise(denoised, stri, ker2, sig)
    #display_result(denoised1, "5", "denoised1")
    return binary

def denoise_binary_denoise_preprocess_image(image, mint, maxt, ker, ker2, sig, stri):
    """
    Загружает изображение, переводит в оттенки серого,
    применяет фиксированный порог и удаляет белый шум морфологией.
    Возвращает очищенное бинарное изображение.
    """
    img = image #cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Не удалось загрузить изображение: {image_path}")
    print(f'\n{"="*60}\nimg: \n{"="*60}\n')
    display_result(img, "1", "img")
    gray = img #cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #display_result(gray, "2", "gray")
    #kernel = np.ones((ker, ker), np.uint8)
    denoised = remove_white_noise(gray, stri, ker2, sig)#cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    print(f'\n{"="*60}\ndenoised: \n{"="*60}\n')
    display_result(denoised, "3", "denoised")
    # ===========================================
    _, binary = cv2.threshold(denoised, mint, maxt, cv2.THRESH_BINARY)
    print(f'\n{"="*60}\nbinary: \n{"="*60}\n')
    display_result(binary, "4", "binary")
    # ===========================================
    denoised1 = remove_white_noise(binary, stri, ker2, sig)
    print(f'\n{"="*60}\ndenoised1: \n{"="*60}\n')
    display_result(denoised1, "5", "denoised1")
    return denoised1

def recognize_text(image, psm=6):
    """
    Распознаёт текст с изображения (numpy array).
    psm (Page Segmentation Mode) — режим разбиения страницы:
        6  – блок с однородным текстом (подходит для нескольких строк)
        11 – произвольный текст (когда нет чёткой структуры)
        8  – одно текстовое слово
    """
    # Конфигурация Tesseract
    custom_config = f'--oem 3 --psm {psm} -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789./- '
    # Белый список символов: буквы (верхний и нижний регистр), цифры, точка, слеш, дефис (часто встречаются в маркировке)
    text = pytesseract.image_to_string(image, config=custom_config, lang='eng')
    # Убираем лишние пробелы и переносы строк
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def main(image):
    print(f"Обработка изображения")
    bedst = ""
    luchee = 100
    for i in range(90, 150, 5): # 
        for j in range(2, 4): # 
            for k in range(2, 4): # 5
                #for r in np.arange(0.0, 0.0, 0.0): # 0.0, 5.0, 0.5
                r = 1.0
                for stri in ["morph_open"]: # "median", , "gaussian"
                    # Предобработка
                    print(f'\n{"="*60}\n threshold {i}; first kernel size {j}; str {stri}; kermel size {k}; sigma {r}\n{"="*60}\n')
                    processed_img = preprocess_image(image, i, 255, j, k, r, stri)

                    cv2.imwrite("processed_for_ocr.png", processed_img)
                    print("Обработанное изображение сохранено как 'processed_for_ocr.png'")

                    results = {}
                    for psm in [6, 11, 8, 7]:
                        text = recognize_text(processed_img, psm)
                        if(len(text) <= 20):
                            results[psm] = text
                            print(f"PSM {psm}: {text}")
                            a = levenshtein_distance(text, "AB31AC")
                            b = levenshtein_distance(text, "SG6105A")
                            c = levenshtein_distance(text, "DYB")
                            if(a < luchee):
                                luchee = a
                                bedst = text
                            if(b < luchee):
                                luchee = b
                                bedst = text
                            if(c < luchee):
                                luchee = c
                                bedst = text
                            print(f'\n a {a}; b {b}; c {c} \n')
                        else:
                            results[psm] = "-"
                            print(f"PSM {psm}: -")

                    processed_img1 = denoise_binary_denoise_preprocess_image(image, i, 255, j, k, r, stri)

                    cv2.imwrite("denoise_binary_denoise_processed_for_ocr.png", processed_img1)
                    print("Обработанное изображение сохранено как 'denoise_binary_denoise_processed_for_ocr.png'")

                    results = {}
                    for psm in [6, 11, 8, 7]:
                        text = recognize_text(processed_img1, psm)
                        if(len(text) <= 20):
                            results[psm] = text
                            print(f"PSM {psm}: {text}")
                            a = levenshtein_distance(text, "AB31AC")
                            b = levenshtein_distance(text, "SG6105A")
                            c = levenshtein_distance(text, "DYB")
                            if(a < luchee):
                                luchee = a
                                bedst = text
                            if(b < luchee):
                                luchee = b
                                bedst = text
                            if(c < luchee):
                                luchee = c
                                bedst = text
                            print(f'\n a {a}; b {b}; c {c} \n')
                        else:
                            results[psm] = "-"
                            print(f"PSM {psm}: -")

                    processed_img2 = denoise_binary_preprocess_image(image, i, 255, j, k, r, stri)

                    cv2.imwrite("denoise_binary_processed_for_ocr.png", processed_img2)
                    print("Обработанное изображение сохранено как 'denoise_binary_processed_for_ocr.png'")

                    results = {}
                    for psm in [6, 11, 8, 7]:
                        text = recognize_text(processed_img1, psm)
                        if(len(text) <= 20):
                            results[psm] = text
                            print(f"PSM {psm}: {text}")
                            a = levenshtein_distance(text, "AB31AC")
                            b = levenshtein_distance(text, "SG6105A")
                            c = levenshtein_distance(text, "DYB")
                            if(a < luchee):
                                luchee = a
                                bedst = text
                            if(b < luchee):
                                luchee = b
                                bedst = text
                            if(c < luchee):
                                luchee = c
                                bedst = text
                            print(f'\n a {a}; b {b}; c {c} \n')
                        else:
                            results[psm] = "-"
                            print(f"PSM {psm}: -")

    print(f'\nbest {bedst}; distance {luchee}\n')
if __name__ == "__main__":
    input_dir = 'p'
    if not os.path.isdir(input_dir):
        print(f'Папка {input_dir} не найдена.')

    files = [f for f in os.listdir(input_dir) if f.lower().endswith('.jpg')]
    if not files:
        print('В папке нет .jpg файлов.')

    for fname in files:
        path = os.path.join(input_dir, fname)
        print(f'\n{"="*60}\nОбрабатываю: {fname}\n{"="*60}')
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f'Не удалось прочитать {fname}')
            continue
        
        main(img)
    